# Étape 1 : préparation des données (SFT et DPO)

Objectif : un corpus médical en français et en anglais, propre et structuré, pour le fine-tuning supervisé (environ 5 000 paires instruction-réponse) et pour l'alignement par préférences (DPO). Des jeux d'évaluation sont mis de côté dès le début et ne servent jamais à l'entraînement.

Sources utilisées :
- MediQAl (`ANR-MALADES/MediQAl`) : questions d'examens médicaux français, licence CC BY 4.0
- FrenchMedMCQA (`nthngdy/frenchmedmcqa`) : QCM d'examens de pharmacie. C'est une copie au format Parquet de l'original (`qanastek/frenchmedmcqa`, Apache 2.0), qui ne se charge plus avec datasets 5 car il dépend d'un script Python
- MedQuAD (`keivalya/MedQuad-MedicalQnADataset`) : questions de patients avec les réponses des sites des NIH, licence CC BY 4.0 pour la version d'origine
- UltraMedical-Preference (`TsinghuaC3I/UltraMedical-Preference`) : paires de réponses classées par GPT-4, licence MIT

L'anonymisation se fait dans le notebook suivant (`02_anonymisation.ipynb`), sur les fichiers produits ici.

In [1]:
import json
import random
import re
import unicodedata
from collections import Counter
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

SEED = 42
VERSION = "1.0"
OUT_DIR = Path("../data/interim")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# le tokenizer du modèle de la mission, pour mesurer les longueurs en tokens
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B-Base")

C:\Users\gregh\Desktop\OpenClassrooms\AI Engineer\P14\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## Journal des transformations

Chaque filtre est noté avec le nombre de lignes avant et après, et la règle appliquée. Le journal est enregistré avec les données, pour qu'on puisse refaire le chemin lors d'un audit.

In [2]:
journal = []


def log_step(source, step, before, after, rule):
    journal.append({"source": source, "etape": step, "avant": before, "apres": after, "regle": rule})
    print(f"{source} | {step} : {before} -> {after}")

In [3]:
def clean_text(text):
    # espaces en trop, lignes vides répétées
    text = unicodedata.normalize("NFC", text or "")
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.splitlines()]
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()


def dedup_key(*parts):
    # clé de comparaison : minuscules, sans accents ni ponctuation
    text = " ".join(p or "" for p in parts).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", " ", text).strip()


def n_tokens(texts):
    return [len(ids) for ids in tokenizer(texts, add_special_tokens=False)["input_ids"]]

## 1. Chargement des corpus

UltraMedical-Preference est lu directement depuis ses fichiers JSON : la conversion automatique du Hub échoue sur ce jeu (« cast error » dans le viewer).

In [4]:
mediqal = {cfg: load_dataset("ANR-MALADES/MediQAl", cfg) for cfg in ("mcqu", "mcqm", "oeq")}
frenchmed = load_dataset("nthngdy/frenchmedmcqa")
medquad = load_dataset("keivalya/MedQuad-MedicalQnADataset")["train"]


def load_ultramedical(split):
    path = hf_hub_download("TsinghuaC3I/UltraMedical-Preference", f"data/{split}.json", repo_type="dataset")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


ultra = {split: load_ultramedical(split) for split in ("train", "dev", "test")}

for cfg, d in mediqal.items():
    print("MediQAl", cfg, {split: len(d[split]) for split in d})
print("FrenchMedMCQA", {split: len(frenchmed[split]) for split in frenchmed})
print("MedQuAD", len(medquad))
print("UltraMedical", {split: len(rows) for split, rows in ultra.items()})

Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


MediQAl mcqu {'train': 10113, 'validation': 2561, 'test': 4343}
MediQAl mcqm {'train': 5767, 'validation': 1466, 'test': 3384}
MediQAl oeq {'test': 4969}
FrenchMedMCQA {'train': 595, 'validation': 164, 'test': 321}
MedQuAD 16407
UltraMedical {'train': 109353, 'dev': 2232, 'test': 777}


## 2. Le schéma des métadonnées

Chaque exemple garde une fiche avec les champs suivants :
- `id` : identifiant stable, construit à partir de la source et de l'identifiant d'origine
- `source`, `sous_source`, `id_origine`, `licence` : d'où vient l'exemple et sous quelle licence
- `langue` : `fr` ou `en`
- `tache` : `qcm_reponse_unique`, `qcm_reponses_multiples`, `question_ouverte`, `question_patient` ou `preference`
- `specialite` et `type_question` : repris de la source quand ils existent
- `origine_annotation` et `niveau_confiance` : qui a validé la réponse, et le niveau de confiance qu'on lui donne
- `symptomes`, `antecedents`, `constantes` : repérés automatiquement dans le cas clinique ou la question
- `version` : version du jeu de données

Le niveau de confiance suit une règle simple :
- `eleve` : correction officielle d'un examen médical (MediQAl, FrenchMedMCQA)
- `moyen` : contenu d'un site de santé public (MedQuAD), ou classement revu par des experts (jeu de test d'UltraMedical d'après ses auteurs)
- `faible` : préférence choisie par GPT-4 sans relecture humaine (entraînement et validation d'UltraMedical)

Symptômes, antécédents et constantes sont repérés avec des listes de mots et des expressions régulières. C'est approximatif : une liste de mots rate les formulations qu'elle ne connaît pas. Pour les antécédents, on ne cherche que dans les phrases qui en parlent (« ATCD », « antécédents », « history of »…), sinon le diagnostic actuel serait compté comme un antécédent.

In [5]:
SYMPTOMS = {
    "fr": ["douleur thoracique", "douleur abdominale", "douleur lombaire", "dyspnée", "essoufflement", "détresse respiratoire",
           "toux", "fièvre", "céphalée", "vomissement", "nausée", "diarrhée", "malaise", "syncope", "perte de connaissance",
           "vertige", "palpitation", "hémoptysie", "hématémèse", "confusion", "convulsion", "paralysie", "déficit moteur",
           "dysurie", "hématurie", "éruption", "prurit", "œdème", "asthénie", "amaigrissement", "sueurs", "cyanose",
           "ictère", "hémorragie", "saignement"],
    "en": ["chest pain", "abdominal pain", "back pain", "shortness of breath", "dyspnea", "respiratory distress", "cough",
           "fever", "headache", "vomiting", "nausea", "diarrhea", "dizziness", "syncope", "fainting", "palpitations",
           "hemoptysis", "confusion", "seizure", "paralysis", "weakness", "numbness", "rash", "itching", "swelling",
           "edema", "fatigue", "weight loss", "sweating", "cyanosis", "jaundice", "bleeding", "sore throat"],
}
HISTORY = {
    "fr": ["diabète", "hypertension", "HTA", "BPCO", "asthme", "insuffisance cardiaque", "infarctus", "AVC", "cancer",
           "insuffisance rénale", "tabagisme", "fumeur", "alcool", "obésité", "épilepsie", "VIH", "hépatite", "cirrhose",
           "dyslipidémie", "allergie"],
    "en": ["diabetes", "hypertension", "COPD", "asthma", "heart failure", "myocardial infarction", "stroke", "cancer",
           "kidney disease", "smoking", "smoker", "alcohol", "obesity", "epilepsy", "HIV", "hepatitis", "cirrhosis",
           "hyperlipidemia", "allergy"],
}
HISTORY_TRIGGERS = {
    "fr": r"atcd|antécédent|connu pour|suivi pour|traité pour|fumeur|tabagi",
    "en": r"history|known|treated for|smok",
}
VITALS = {
    "fr": {
        "fc": r"\b(?:FC|fréquence cardiaque|pouls)\s*(?:=|:|est à|est de|à|de)?\s*(\d{2,3})",
        "pa": r"\b(?:PA|TA|pression artérielle|tension artérielle)\s*(?:=|:|est à|est de|à|de)?\s*(\d{2,3}\s*/\s*\d{2,3})",
        "fr": r"\b(?:FR|fréquence respiratoire)\s*(?:=|:|est à|est de|à|de)?\s*(\d{1,2})\b",
        "spo2": r"\b(?:SpO2|SaO2|saturation(?: en oxygène)?)\s*(?:=|:|est à|est de|à|de)?\s*(\d{2,3})\s*%",
        "temperature": r"\b(?:T°?|température)\s*(?:=|:|est à|est de|à|de)?\s*(\d{2}(?:\s?[.,]\s?\d)?)\s*°\s*(\d)?",
    },
    "en": {
        "fc": r"\b(?:pulse|heart rate|HR)\s*(?:is|of|was|:|=)?\s*(\d{2,3})",
        "pa": r"\b(?:blood pressure|BP)\s*(?:is|of|was|:|=)?\s*(\d{2,3}\s*/\s*\d{2,3})",
        "fr": r"\b(?:respirations|respiratory rate|RR)\s*(?:are|is|of|was|:|=)?\s*(\d{1,2})\b",
        "spo2": r"\b(?:oxygen saturation|O2 saturation|SpO2|pulse oximetry)\D{0,25}?(\d{2,3})\s*%",
        "temperature": r"\b(?:temperature|temp)\s*(?:is|of|was|:|=)?\s*(\d{2,3}(?:\.\d)?)\s*°?\s*([CF])\b",
    },
}


def find_terms(text, terms):
    found = []
    for term in terms:
        # pluriels simples acceptés (« vomissements », « seizures »)
        if re.search(r"\b" + re.escape(term) + r"s?\b", text, flags=re.IGNORECASE):
            found.append(term.lower())
    return found


def extract_clinical(text, lang):
    text = text or ""
    history_text = " ".join(s for s in re.split(r"(?<=[.;\n])", text)
                            if re.search(HISTORY_TRIGGERS[lang], s, flags=re.IGNORECASE))
    vitals = {}
    for name, pattern in VITALS[lang].items():
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if not m:
            vitals[name] = None
            continue
        value = re.sub(r"\s+", "", m.group(1)).replace(",", ".")
        if name == "temperature":
            extra = m.group(2) if m.lastindex == 2 else None
            if extra and extra.isdigit():
                # notation française « 39°5 » pour 39,5 degrés
                value, extra = value + "." + extra, None
            # unité notée en anglais (°C ou °F), toujours °C dans les cas français
            value += " °" + (extra.upper() if extra else "C")
        vitals[name] = value
    return {"symptomes": find_terms(text, SYMPTOMS[lang]),
            "antecedents": find_terms(history_text, HISTORY[lang]),
            "constantes": vitals}


# test sur le premier cas de médecine d'urgence des questions ouvertes de MediQAl
case = next(r["clinical_case"] for r in mediqal["oeq"]["test"] if r["medical_subject"] == "Emergency Medicine")
print(case[:500])
print(extract_clinical(case, "fr"))

Homme, 58 ans, majoration de dyspnée chez un BPCO connu depuis 20 ans.

 Examen d’entrée : 
 Constantes :
 - FC = 130bpm
 - PA = 130/80 aux deux bras
 - FR = 40/min
 - SpO2 = 80%
 - T = 38 ,5°C
 Tirage sus-sternal
 Cyanose
 Sueurs
 Auscultation : 
 - Ronchi bilatéraux et sibilants
 - Tympanisme bilatéral
 Hépatomégalie douloureuse
 Reflux hépato-jugulaire
 Turgescence jugulaire
 Hippocratisme digital
 Cyanose unguéale
 
 ATCD 
 Fumeur (80 PA)
 BPCO
 
 Traitement habituel
 β2 mimétiques 
 
 HDM :
{'symptomes': ['dyspnée', 'sueurs', 'cyanose'], 'antecedents': ['fumeur'], 'constantes': {'fc': '130', 'pa': '130/80', 'fr': '40', 'spo2': '80', 'temperature': '38.5 °C'}}


In [6]:
def make_row(source, sub_source, origin_id, licence, lang, task, prompt, answer, specialty, qtype,
             annotation, confidence, context):
    meta = extract_clinical(context, lang)
    return {
        "id": f"{source}-{sub_source}-{origin_id}".lower().replace(" ", "_"),
        "messages": [{"role": "user", "content": prompt}, {"role": "assistant", "content": answer}],
        "source": source, "sous_source": sub_source, "id_origine": str(origin_id), "licence": licence,
        "langue": lang, "tache": task, "specialite": specialty, "type_question": qtype,
        "origine_annotation": annotation, "niveau_confiance": confidence,
        "symptomes": meta["symptomes"], "antecedents": meta["antecedents"], "constantes": meta["constantes"],
        "version": VERSION,
    }

## 3. Des QCM aux paires question-réponse

Format retenu pour la réponse attendue : la lettre puis le texte de la proposition (« Réponse : A, le vaccin contre la fièvre jaune est obligatoire »). On n'invente rien, le modèle apprend à répondre par une phrase, et la lettre permet de noter automatiquement les réponses à l'évaluation.

La consigne reprend le cas clinique quand il existe, la question, les propositions, et précise s'il y a une ou plusieurs bonnes réponses.

In [7]:
LETTERS = "ABCDE"


def lower_first(text):
    # « Le vaccin » devient « le vaccin », mais « VIH » reste « VIH »
    if len(text) > 1 and text[0].isupper() and text[1].islower():
        return text[0].lower() + text[1:]
    return text


def mcq_prompt(case, question, options, multiple):
    parts = [f"Cas clinique : {case}"] if case else []
    parts.append(f"Question : {question}")
    parts.append("\n".join(f"{letter}. {opt}" for letter, opt in zip(LETTERS, options) if opt))
    parts.append("Plusieurs réponses peuvent être exactes." if multiple else "Une seule réponse est exacte.")
    return "\n\n".join(parts)


def mcq_answer(letters, options):
    texts = dict(zip(LETTERS, options))
    if len(letters) == 1:
        return f"Réponse : {letters[0]}, {lower_first(texts[letters[0]])}"
    listed = ", ".join(letters[:-1]) + " et " + letters[-1]
    return f"Réponses : {listed}\n" + "\n".join(f"- {letter}, {lower_first(texts[letter])}" for letter in letters)


def mediqal_mcq_rows(cfg, split):
    rows, dropped = [], 0
    for r in mediqal[cfg][split]:
        options = [clean_text(r[f"answer_{x}"]) for x in "abcde"]
        letters = [x.strip() for x in r["correct_answers"].split(",") if x.strip()]
        if not letters or any(not options[LETTERS.index(x)] for x in letters):
            dropped += 1
            continue
        case, question = clean_text(r["clinical_case"]), clean_text(r["question"])
        rows.append(make_row(
            "mediqal", f"{cfg}-{split}", r["id"], "CC BY 4.0", "fr",
            "qcm_reponse_unique" if cfg == "mcqu" else "qcm_reponses_multiples",
            mcq_prompt(case, question, options, cfg == "mcqm"), mcq_answer(letters, options),
            r["medical_subject"], r["question_type"], "correction officielle d'examen", "eleve", case + " " + question))
    if dropped:
        print(f"{cfg} {split} : {dropped} questions sans proposition correspondant à la bonne réponse")
    return rows


def frenchmed_rows(split):
    rows = []
    for r in frenchmed[split]:
        options = [clean_text(r[f"answer_{x}"]) for x in "abcde"]
        letter = LETTERS[r["correct_answers"]]
        question = clean_text(r["question"])
        rows.append(make_row(
            "frenchmedmcqa", split, r["id"][:16], "Apache 2.0 (original)", "fr", "qcm_reponse_unique",
            mcq_prompt("", question, options, False), mcq_answer([letter], options),
            "Pharmacy", None, "correction officielle d'examen", "eleve", question))
    return rows


def oeq_rows():
    rows = []
    for r in mediqal["oeq"]["test"]:
        case, question = clean_text(r["clinical_case"]), clean_text(r["question"])
        prompt = f"Cas clinique : {case}\n\nQuestion : {question}" if case else question
        rows.append(make_row(
            "mediqal", "oeq", f"{r['id']}-{r['cc_question_number']}", "CC BY 4.0", "fr", "question_ouverte",
            prompt, clean_text(r["answer"]), r["medical_subject"], r["question_type"],
            "correction officielle d'examen", "eleve", case + " " + question))
    return rows


example = mediqal_mcq_rows("mcqu", "train")[0]
print(example["messages"][0]["content"])
print("---")
print(example["messages"][1]["content"])

Cas clinique : Monsieur R. âgé de 37 ans, forestier de son métier, doit partir travailler pendant 2 ans au Gabon dans la région de Lambaréné. Monsieur R. a l'intention de se faire accompagner de son épouse âgée de 36 ans et de son fils âgé de 5 ans. Ils logeront à quelques kilomètres de la forêt mais Mr R., de par son métier, passera la plupart de son temps au milieu des bois. Ils consultent 4 mois avant le départ pour se renseigner au sujet des vaccinations et de la chimioprophylaxie du paludisme. Vous lui expliquez les conseils d'hygiène, les éléments de prévention essentiels et les différents modes de contamination des maladies présentes dans la région

Question : Au sujet des vaccinations :

A. Le vaccin contre la fièvre jaune est obligatoire
B. Le vaccin contre la fièvre jaune peut se faire au cabinet du médecin traitant
C. Le vaccin contre le choléra est un vaccin qui assure une protection supérieure à 80%
D. Le vaccin contre l'hépatite B est conseillé mais il ne se trouve pas en

## 4. Les jeux d'évaluation, mis de côté en premier

On les construit avant le jeu d'entraînement, pour pouvoir retirer de l'entraînement tout ce qui leur ressemble :
- `eval_mediqal_qcu` : 500 QCM à réponse unique tirés du test officiel de MediQAl
- `eval_mediqal_qcm` : 300 QCM à réponses multiples tirés du test officiel de MediQAl
- `eval_frenchmedmcqa` : le test officiel de FrenchMedMCQA
- `eval_ultramedical` : les 777 paires de test d'UltraMedical, dont le classement a été revu par des experts selon les auteurs

On note les clés de tout le test officiel de MediQAl et de FrenchMedMCQA, pas seulement des questions tirées : une question du test ne doit jamais servir à l'entraînement.

In [8]:
def drop_duplicates(rows, source):
    seen, out = set(), []
    for r in rows:
        k = dedup_key(r["messages"][0]["content"])
        if k not in seen:
            seen.add(k)
            out.append(r)
    log_step(source, "doublons", len(rows), len(out), "même consigne (casse, accents et ponctuation ignorés)")
    return out


qcu_test = drop_duplicates(mediqal_mcq_rows("mcqu", "test"), "mediqal-mcqu-test")
qcm_test = drop_duplicates(mediqal_mcq_rows("mcqm", "test"), "mediqal-mcqm-test")
fm_test = drop_duplicates(frenchmed_rows("test"), "frenchmedmcqa-test")

eval_keys = {dedup_key(r["messages"][0]["content"]) for r in qcu_test + qcm_test + fm_test}

rng = random.Random(SEED)
eval_sets = {
    "eval_mediqal_qcu": rng.sample(qcu_test, 500),
    "eval_mediqal_qcm": rng.sample(qcm_test, 300),
    "eval_frenchmedmcqa": fm_test,
}
for name, rows in eval_sets.items():
    print(name, len(rows), "| spécialités :", len(Counter(r["specialite"] for r in rows)))

mediqal-mcqu-test | doublons : 4343 -> 4279


mediqal-mcqm-test | doublons : 3384 -> 3338
frenchmedmcqa-test | doublons : 321 -> 321


eval_mediqal_qcu 500 | spécialités : 30
eval_mediqal_qcm 300 | spécialités : 25
eval_frenchmedmcqa 321 | spécialités : 1


## 5. Le jeu SFT

Composition visée : environ 5 000 paires, moitié en français, moitié en anglais.

En français (2 500) :
- 1 300 QCM à réponse unique et 600 QCM à réponses multiples de MediQAl (partie entraînement)
- 300 questions ouvertes de MediQAl : toutes celles de médecine d'urgence et de réanimation, les plus proches du triage, complétées au hasard
- 300 QCM de FrenchMedMCQA (partie entraînement)

En anglais (2 500) : des questions de MedQuAD, en gardant une seule réponse par question et les types de questions cliniques. Les types centrés sur la génétique (hérédité, mutations, fréquence dans la population) et sur la recherche sont écartés, ils sont loin du triage.

Filtres communs : pas de doublon, pas de question présente dans un jeu d'évaluation, et 1 024 tokens au plus pour la consigne et la réponse réunies.

In [9]:
def remove_eval_overlap(rows, source):
    out = [r for r in rows if dedup_key(r["messages"][0]["content"]) not in eval_keys]
    log_step(source, "recouvrement avec l'évaluation", len(rows), len(out), "consigne présente dans un jeu de test officiel")
    return out


def keep_short(rows, source, limit=1024):
    lengths = n_tokens([r["messages"][0]["content"] + "\n" + r["messages"][1]["content"] for r in rows])
    out = [r for r, n in zip(rows, lengths) if n <= limit]
    log_step(source, "longueur", len(rows), len(out), f"consigne + réponse <= {limit} tokens Qwen3")
    return out


qcu_pool = keep_short(remove_eval_overlap(drop_duplicates(mediqal_mcq_rows("mcqu", "train"), "mediqal-mcqu-train"), "mediqal-mcqu-train"), "mediqal-mcqu-train")
qcm_pool = keep_short(remove_eval_overlap(drop_duplicates(mediqal_mcq_rows("mcqm", "train"), "mediqal-mcqm-train"), "mediqal-mcqm-train"), "mediqal-mcqm-train")
fm_pool = keep_short(remove_eval_overlap(drop_duplicates(frenchmed_rows("train"), "frenchmedmcqa-train"), "frenchmedmcqa-train"), "frenchmedmcqa-train")
oeq_pool = keep_short(drop_duplicates(oeq_rows(), "mediqal-oeq"), "mediqal-oeq")

mediqal-mcqu-train | doublons : 10113 -> 9895


mediqal-mcqu-train | recouvrement avec l'évaluation : 9895 -> 9872


mediqal-mcqu-train | longueur : 9872 -> 9872


mediqal-mcqm-train | doublons : 5767 -> 5672


mediqal-mcqm-train | recouvrement avec l'évaluation : 5672 -> 5641


mediqal-mcqm-train | longueur : 5641 -> 5641
frenchmedmcqa-train | doublons : 595 -> 592
frenchmedmcqa-train | recouvrement avec l'évaluation : 592 -> 591
frenchmedmcqa-train | longueur : 591 -> 591


mediqal-oeq | doublons : 4969 -> 4941


mediqal-oeq | longueur : 4941 -> 4905


In [10]:
urgent = [r for r in oeq_pool if r["specialite"] in ("Emergency Medicine", "Intensive Care")]
others = [r for r in oeq_pool if r["specialite"] not in ("Emergency Medicine", "Intensive Care")]
rng = random.Random(SEED)
oeq_selected = urgent + rng.sample(others, 300 - len(urgent))
print("questions ouvertes : urgence et réanimation", len(urgent), "+ autres", 300 - len(urgent))

sft_fr = rng.sample(qcu_pool, 1300) + rng.sample(qcm_pool, 600) + oeq_selected + rng.sample(fm_pool, 300)
print("français :", len(sft_fr))

questions ouvertes : urgence et réanimation 254 + autres 46
français : 2500


In [11]:
CLINICAL_QTYPES = {"information", "symptoms", "treatment", "causes", "exams and tests", "outlook", "susceptibility",
                   "considerations", "prevention", "stages", "complications"}


def fix_question(q):
    # « Who is at risk for X? ? » devient « Who is at risk for X? »
    return re.sub(r"\?(\s*\?)+", "?", clean_text(q))


medquad_rows = []
for i, r in enumerate(medquad):
    question, answer = fix_question(r["Question"]), clean_text(r["Answer"])
    if not question or not answer:
        continue
    medquad_rows.append(make_row(
        "medquad", r["qtype"], i, "CC BY 4.0 (original)", "en", "question_patient", question, answer,
        None, r["qtype"], "site de santé public (NIH)", "moyen", question))

before = len(medquad)
medquad_rows = drop_duplicates(medquad_rows, "medquad")
kept = [r for r in medquad_rows if r["type_question"] in CLINICAL_QTYPES]
log_step("medquad", "types cliniques", len(medquad_rows), len(kept), "types génétique, fréquence et recherche écartés")
medquad_pool = keep_short(kept, "medquad")

rng = random.Random(SEED)
sft_en = rng.sample(medquad_pool, 2500)
print("anglais :", len(sft_en))
print(Counter(r["type_question"] for r in sft_en).most_common())

medquad | doublons : 16407 -> 14298
medquad | types cliniques : 14298 -> 10440


medquad | longueur : 10440 -> 10073
anglais : 2500
[('information', 803), ('symptoms', 690), ('treatment', 486), ('causes', 152), ('exams and tests', 145), ('outlook', 77), ('prevention', 55), ('susceptibility', 45), ('considerations', 24), ('complications', 13), ('stages', 10)]


Découpage du jeu SFT en entraînement, validation et test : 90 %, 5 % et 5 %, source par source, pour que chaque partie garde le même mélange.

In [12]:
def split_by_source(rows, val_frac=0.05, test_frac=0.05):
    rng = random.Random(SEED)
    parts = {"train": [], "val": [], "test": []}
    for source in sorted({r["source"] + r["tache"] for r in rows}):
        group = [r for r in rows if r["source"] + r["tache"] == source]
        rng.shuffle(group)
        n_val, n_test = round(len(group) * val_frac), round(len(group) * test_frac)
        parts["val"] += group[:n_val]
        parts["test"] += group[n_val:n_val + n_test]
        parts["train"] += group[n_val + n_test:]
    return parts


sft = split_by_source(sft_fr + sft_en)
for split, rows in sft.items():
    print(split, len(rows), dict(Counter(r["langue"] for r in rows)))

train 4500 {'fr': 2250, 'en': 2250}
val 250 {'fr': 125, 'en': 125}
test 250 {'fr': 125, 'en': 125}


## 6. Le jeu DPO

Le jeu DPO vient d'UltraMedical-Preference (partie entraînement, environ 109 000 paires). Filtres appliqués, dans l'ordre :
1. On retire les questions qui apparaissent aussi dans la validation ou le test d'UltraMedical (le jeu en partage quelques-unes entre ses parties), puis celles qui viennent de MedQuAD : elles pourraient recouper notre jeu SFT anglais.
2. On garde les paires où la réponse préférée a au moins 1 point de plus que l'autre sur 5. Quand les deux notes sont égales, ou quand la réponse « préférée » a une note plus basse, la paire n'apprend rien de fiable.
3. Consigne et réponse la plus longue : 1 024 tokens au plus.
4. Une seule paire par question.

Un dernier point vient de l'exploration : dans les catégories « easy » et « hard », la réponse préférée est la plus longue dans environ 8 cas sur 10. Un DPO sur ces paires risque d'apprendre que plus long veut dire meilleur. On équilibre donc le jeu final : dans la moitié des paires la réponse préférée est la plus longue, dans l'autre moitié la plus courte.

In [13]:
def ultra_source(r):
    return r["prompt_id"].split(",")[0]


def ultra_gap(r):
    return r["metadata"]["chosen"]["score"] - r["metadata"]["rejected"]["score"]


def filter_ultra(rows, name, excluded_keys):
    out = [r for r in rows if dedup_key(r["prompt"]) not in excluded_keys]
    log_step(name, "recouvrement", len(rows), len(out), "question présente dans une partie réservée à la validation ou au test")
    before, out = len(out), [r for r in out if ultra_source(r) != "MedQuad"]
    log_step(name, "questions MedQuAD", before, len(out), "source MedQuad retirée (recouvrement possible avec le SFT)")
    before, out = len(out), [r for r in out if ultra_gap(r) >= 1]
    log_step(name, "écart de note", before, len(out), "note préférée - note rejetée >= 1 (sur 5)")
    lp = n_tokens([r["prompt"] for r in out])
    lc = n_tokens([r["chosen"][1]["content"] for r in out])
    lr = n_tokens([r["rejected"][1]["content"] for r in out])
    for r, p, c, rj in zip(out, lp, lc, lr):
        r["_longest"], r["_chosen_longer"] = p + max(c, rj), c > rj
    before, out = len(out), [r for r in out if r["_longest"] <= 1024]
    log_step(name, "longueur", before, len(out), "consigne + réponse la plus longue <= 1024 tokens Qwen3")
    seen, unique = set(), []
    for r in out:
        k = dedup_key(r["prompt"])
        if k not in seen:
            seen.add(k)
            unique.append(r)
    log_step(name, "doublons", len(out), len(unique), "même question")
    return unique


# UltraMedical partage quelques questions entre ses propres parties : on les retire de l'entraînement
test_keys = {dedup_key(r["prompt"]) for r in ultra["test"]}
dev_keys = {dedup_key(r["prompt"]) for r in ultra["dev"]}
ultra_train = filter_ultra(ultra["train"], "ultramedical-train", test_keys | dev_keys)
ultra_dev = filter_ultra(ultra["dev"], "ultramedical-dev", test_keys)
print("réponse préférée plus longue :", f"{100 * sum(r['_chosen_longer'] for r in ultra_train) / len(ultra_train):.0f} %")
for label in ("easy", "hard", "length"):
    group = [r for r in ultra_train if r["label_type"] == label]
    share = 100 * sum(r["_chosen_longer"] for r in group) / len(group)
    print(f"  catégorie {label:6s} : {len(group)} paires, réponse préférée plus longue dans {share:.0f} % des cas")

ultramedical-train | recouvrement : 109353 -> 107617
ultramedical-train | questions MedQuAD : 107617 -> 101874
ultramedical-train | écart de note : 101874 -> 95612


ultramedical-train | longueur : 95612 -> 93013


ultramedical-train | doublons : 93013 -> 66219
ultramedical-dev | recouvrement : 2232 -> 2231
ultramedical-dev | questions MedQuAD : 2231 -> 2125
ultramedical-dev | écart de note : 2125 -> 1984


ultramedical-dev | longueur : 1984 -> 1928
ultramedical-dev | doublons : 1928 -> 1915
réponse préférée plus longue : 69 %
  catégorie easy   : 15493 paires, réponse préférée plus longue dans 90 % des cas
  catégorie hard   : 31111 paires, réponse préférée plus longue dans 79 % des cas
  catégorie length : 19615 paires, réponse préférée plus longue dans 37 % des cas


In [14]:
def balanced_sample(rows, n, seed=SEED):
    rng = random.Random(seed)
    longer = [r for r in rows if r["_chosen_longer"]]
    shorter = [r for r in rows if not r["_chosen_longer"]]
    return rng.sample(longer, n // 2) + rng.sample(shorter, n // 2)


def dpo_row(r, split, annotation, confidence):
    md = r["metadata"]
    meta = extract_clinical(r["prompt"], "en")
    return {
        "id": f"ultramedical-{split}-{r['prompt_id']}".lower().replace(" ", "_").replace(",", "-"),
        "prompt": [{"role": "user", "content": clean_text(r["prompt"])}],
        "chosen": [{"role": "assistant", "content": clean_text(r["chosen"][1]["content"])}],
        "rejected": [{"role": "assistant", "content": clean_text(r["rejected"][1]["content"])}],
        "source": "ultramedical", "sous_source": ultra_source(r), "id_origine": r["prompt_id"], "licence": "MIT",
        "langue": "en", "tache": "preference", "specialite": None, "type_question": r["label_type"],
        "origine_annotation": annotation, "niveau_confiance": confidence,
        "symptomes": meta["symptomes"], "antecedents": meta["antecedents"], "constantes": meta["constantes"],
        "note_choisie": md["chosen"]["score"], "note_rejetee": md["rejected"]["score"],
        "modele_choisi": md["chosen"]["model"], "modele_rejete": md["rejected"]["model"],
        "version": VERSION,
    }


dpo = {
    "train": [dpo_row(r, "train", "classement GPT-4", "faible") for r in balanced_sample(ultra_train, 3000)],
    "val": [dpo_row(r, "dev", "classement GPT-4", "faible") for r in balanced_sample(ultra_dev, 300)],
}
eval_sets["eval_ultramedical"] = [dpo_row(r, "test", "classement revu par des experts (selon les auteurs)", "moyen")
                                  for r in ultra["test"]]
for split, rows in dpo.items():
    print(split, len(rows), Counter(r["sous_source"] for r in rows).most_common(4))
print("eval_ultramedical", len(eval_sets["eval_ultramedical"]),
      Counter(r["type_question"] for r in eval_sets["eval_ultramedical"]))

train 3000 [('WikiInstruct', 790), ('MedMCQA', 486), ('TextBookQA', 478), ('MedQA', 333)]
val 300 [('WikiInstruct', 69), ('TextBookQA', 52), ('MedMCQA', 51), ('ChatDoctor', 39)]
eval_ultramedical 777 Counter({'easy': 238, 'hard': 196, 'length': 180, 'human': 163})


## 7. Contrôles

On vérifie qu'aucune consigne n'apparaît à la fois dans l'entraînement et dans une partie réservée à l'évaluation, et on regarde ce que les listes de mots ont réussi à repérer.

In [15]:
def keys(rows, field="messages"):
    if field == "messages":
        return {dedup_key(r["messages"][0]["content"]) for r in rows}
    return {dedup_key(r["prompt"][0]["content"]) for r in rows}


all_eval = keys(eval_sets["eval_mediqal_qcu"] + eval_sets["eval_mediqal_qcm"] + eval_sets["eval_frenchmedmcqa"])
checks = {
    "SFT train / SFT val": len(keys(sft["train"]) & keys(sft["val"])),
    "SFT train / SFT test": len(keys(sft["train"]) & keys(sft["test"])),
    "SFT (tout) / jeux d'évaluation": len(keys(sft["train"] + sft["val"] + sft["test"]) & all_eval),
    "DPO train / DPO val": len(keys(dpo["train"], "prompt") & keys(dpo["val"], "prompt")),
    "DPO train / eval UltraMedical": len(keys(dpo["train"], "prompt") & keys(eval_sets["eval_ultramedical"], "prompt")),
    "SFT (tout) / DPO train": len(keys(sft["train"] + sft["val"] + sft["test"]) & keys(dpo["train"], "prompt")),
}
for name, n in checks.items():
    print(f"{name} : {n} consigne(s) en commun")

SFT train / SFT val : 0 consigne(s) en commun
SFT train / SFT test : 0 consigne(s) en commun
SFT (tout) / jeux d'évaluation : 0 consigne(s) en commun
DPO train / DPO val : 0 consigne(s) en commun
DPO train / eval UltraMedical : 0 consigne(s) en commun
SFT (tout) / DPO train : 0 consigne(s) en commun


In [16]:
def coverage(rows):
    df = pd.DataFrame({
        "source": [r["source"] for r in rows],
        "symptomes": [bool(r["symptomes"]) for r in rows],
        "antecedents": [bool(r["antecedents"]) for r in rows],
        "constantes": [any(v for v in r["constantes"].values()) for r in rows],
    })
    return (df.groupby("source")[["symptomes", "antecedents", "constantes"]].mean() * 100).round(1)


print("part des exemples (%) où au moins un élément a été repéré")
coverage(sft["train"] + dpo["train"])

part des exemples (%) où au moins un élément a été repéré


,symptomes,antecedents,constantes
source,,,
frenchmedmcqa,0.7,0.0,0.0
mediqal,29.7,7.9,16.1
medquad,1.9,0.2,0.0
ultramedical,24.3,6.6,6.0


In [17]:
# contrôle à la main : les valeurs repérées, et le passage du texte d'où elles viennent
KEYWORDS = r"\b(?:FC|pouls|fréquence cardiaque|pulse|heart rate|PA|TA|pression artérielle|blood pressure|SpO2|SaO2|température|temperature)\b"
pool = [r for r in sft["train"] + dpo["train"] if any(r["constantes"].values())]
print(len(pool), "exemples avec au moins une constante repérée. Échantillon :")
for r in random.Random(SEED).sample(pool, 8):
    text = r["messages"][0]["content"] if "messages" in r else r["prompt"][0]["content"]
    m = re.search(KEYWORDS, text, flags=re.IGNORECASE)
    print({k: v for k, v in r["constantes"].items() if v}, "|", " ".join(text[m.start():m.start() + 110].split()))

499 exemples avec au moins une constante repérée. Échantillon :
{'fc': '72', 'pa': '136/84', 'fr': '18', 'temperature': '38.1 °C'} | temperature is 38.1°C (100.6°F), pulse is 72/min, respirations are 18/min, and blood pressure is 136/84 mm Hg.
{'temperature': '40 °C'} | température est à 40°C, avec présence d'un herpès de la lèvre supérieure. A la percussion on retrouve une mati
{'temperature': '37 °C'} | température est à 37°C Question : Quel est le moyen d'obtenir avec certitude le diagnostic probable de cancer
{'pa': '100/60'} | blood pressure of 100/60 mm Hg but still symptomatic, what is the optimal site for chest tube placement to pre
{'fc': '110', 'temperature': '39.5 °C'} | température est à 39°5 et l'enfant est particulièrement abattue et somnolente. L'examen clinique vous permet d
{'fc': '160', 'temperature': '40 °C'} | température est à 40°C; son pouls à 160/min, sa T.A. à 9/6. La crise cède après 15 minutes. Il persiste un déf
{'temperature': '38.2 °C'} | température à 38°2

Les valeurs sont recopiées telles qu'elles sont écrites dans la source, sans contrôle de vraisemblance. Pendant les essais, un cas de MediQAl indiquait « FC 24/mn » pour une enfant de 12 ans, sans doute une faute de frappe de la source. On ne corrige pas les données d'origine, la limite est notée dans la documentation.

## 8. Enregistrement

Les fichiers sont écrits au format JSONL (une ligne par exemple) dans `data/interim/`, avec le journal des transformations. Ils ne sont pas encore anonymisés : c'est l'objet du notebook suivant.

In [18]:
def write_jsonl(rows, name):
    path = OUT_DIR / f"{name}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps({k: v for k, v in r.items() if not k.startswith("_")}, ensure_ascii=False) + "\n")
    return len(rows)


files = {f"sft_{split}": rows for split, rows in sft.items()}
files.update({f"dpo_{split}": rows for split, rows in dpo.items()})
files.update(eval_sets)
for name, rows in files.items():
    print(f"{name}.jsonl : {write_jsonl(rows, name)} lignes")

with open(OUT_DIR / "journal_transformations.json", "w", encoding="utf-8") as f:
    json.dump({"version": VERSION, "seed": SEED, "etapes": journal}, f, ensure_ascii=False, indent=1)
print(len(journal), "étapes dans le journal")

sft_train.jsonl : 4500 lignes
sft_val.jsonl : 250 lignes
sft_test.jsonl : 250 lignes
dpo_train.jsonl : 3000 lignes
dpo_val.jsonl : 300 lignes
eval_mediqal_qcu.jsonl : 500 lignes
eval_mediqal_qcm.jsonl : 300 lignes
eval_frenchmedmcqa.jsonl : 321 lignes


eval_ultramedical.jsonl : 777 lignes
27 étapes dans le journal
